### Install environment

In [ ]:
import importlib.util
if importlib.util.find_spec("flashrank") is None:
    !pip install vllm==0.10.0
    !pip install triton==3.2.0
    !pip install flashrank langchain langchain-community langchain_google_genai openai faiss-cpu langchain_huggingface crawl4ai unidecode pymupdf4llm google-genai rapidfuzz
    !pip uninstall -y openai
    !pip install openai==1.90.0
else:
    print("All libs aldready installed")

In [ ]:
!wget -q -O ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip -o ngrok.zip
!mv ngrok /usr/local/bin/ngrok
!chmod +x /usr/local/bin/ngrok
!ps aux | grep ngrok

In [ ]:
import os
# Fixed, do not change
os.environ["GLOO_SOCKET_NAME"] = "eth0"
os.environ["NCCL_SOCKET_NAME"] = "eth0"
os.environ["VLLM_HOST_IP"] = "127.0.0.1" # Internal ip for data communicate between VLLM components
os.environ["VLLM_USE_V1"] = "0" # T4 have compute capacity of 7.5, it need at least 8.0 to use V1

##### Download package from server

In [ ]:
DOMAIN = "https://d8ede3aed5bb.ngrok-free.app"
BASE_PATH = "/kaggle/working/"

In [ ]:
import requests
import io
import tarfile
import shutil
def unpack_folder(data: bytes, path: str):
    if os.path.exists(path): # Remove old code
        shutil.rmtree(path)
    with io.BytesIO(data) as tar_buffer:
        with tarfile.open(fileobj=tar_buffer, mode='r:gz') as tar:
            tar.extractall(path=path)
def unpack_file(data: bytes, path: str):
    if os.path.exists(path):
        os.remove(path)
    with open(path, 'wb') as file:
        file.write(data)
def unpack_list(*names: str):
    if DOMAIN == "http://127.0.0.1:8000": return
    for name in names:
        if "." in name:
            url = f"{DOMAIN}/package/{name}"
        else:
            url = f"{DOMAIN}/package/{name}"
        data = requests.get(url).content
        if "." in name:
            unpack_file(data, name)
        else:
            unpack_folder(data, name)
unpack_list(
    "data_retriever", "server", "static.pkl", "worker.env", "instruction", "school_mapper", "school_name.json",
    "lora/reader_v1"
)

### Config

Load environment variable (api keys)

In [ ]:
from dotenv import load_dotenv
load_dotenv(f"{BASE_PATH}worker.env")

In [ ]:
import shutil, os
if os.path.exists(f"{BASE_PATH}/data_logs"):
    shutil.rmtree(f"{BASE_PATH}/data_logs")

Setup ngrok

In [ ]:
NGROK_PORT = 8002
if DOMAIN != "http://127.0.0.1:8000":
    import subprocess
    subprocess.run(["ngrok", "config", "add-authtoken", os.getenv("NGROK_TOKEN_1", "")])
    subprocess.Popen(["ngrok", "http", str(NGROK_PORT)], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

Login hugging face

In [ ]:
cmd = [
    "hf", "auth", "login",
    "--token", os.getenv("HUGGING_FACE_TOKEN")
]
import subprocess
subprocess.run(cmd)

##### Config

In [ ]:
from data_retriever import *
from server import *
from school_mapper import SchoolMapper
from typing import AsyncGenerator, NotRequired, Protocol
from typing import Callable, AsyncGenerator
import os
import pickle
import json
import asyncio
import enum
import traceback
import copy

In [ ]:
from vllm.lora.request import LoRARequest

In [ ]:
from typing import Protocol, AsyncGenerator, TypedDict
class KeywordInfo(TypedDict):
    query: str
    priority: float
    info: str
    school: str
class KeywordModelProtocol(Protocol):
    async def keywords(self, question: str, params: GenerationParams, threshold: float = 0.5) -> list[KeywordInfo]: ...

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"
# Retriever config
search_config = WebsearchConfig(
    page_timeout=15,
    file_timeout=15,
)
rag_config = RagConfig(
    embedding_name="intfloat/multilingual-e5-small",
    device="cuda"
)
splitter_config = SplitterConfig(
    tokenizer_name=MODEL_ID,
    chunk_size=512,
    chunk_overlap=0,
    # device="cuda"
)
table_merge_config = MergeTableConfig(
    k_max_previous=5,
    k_max_next=5
)
neighbor_config = MergeNeighborConfig(
    k_previous_chunks=1,
    k_next_chunks=1
)
# Sampling Params
PAGE_RERANKER_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 4096
}
KEYWORDS_PARAMS = {
    "temperature": 0.5,
    "top_p": 0.9,
    "max_tokens": 4096
}
SEP = "$$$"
MODELS: list[ModelInfo] = [
    {
        "name": "Qwen3 4B",
        "id": "Qwen/Qwen3-4B"
    },
    {
        "name": "Qwen3 4B LoRA",
        "id": f"Qwen/Qwen3-4B{SEP}1"
    }
]
CLIENT_INFO: WorkerServerInfo = {
    "name": "Test Reader only",
    "domain": "http://127.0.0.1:8002", # Auto change when run with ngrok
    "models": MODELS
}
READER_LORA = LoRARequest(
    lora_int_id= 1,
    lora_name= "Qwen Reader Lora",
    lora_path= f"{BASE_PATH}lora/reader_v1"
)
LORA_MAP = {
    1: READER_LORA
}

##### Template, instruction, prefix

In [ ]:
from instruction import *

### Utility class

##### Data Retriever

In [ ]:
class Retriever:
    """Search in web"""
    def __init__(self, llm_ranker: PageRerankModelProtocol, llm_keywords: KeywordModelProtocol) -> None:
        self.pipeline = DataRetrieverPipeline(
            llm_ranker,
            websearch_config=search_config,
            rag_config=rag_config,
            splitter_config=splitter_config,
            neighbor_merge_config=neighbor_config,
            table_merge_config=table_merge_config
        )
        self.llm_keywords = llm_keywords
        self.school_mapper = SchoolMapper(f"{BASE_PATH}school_name.json")
    async def start(self):
        """Initialize websearch"""
        await self.pipeline.start()
    async def retrive(self, question: str, params: GenerationParams) -> tuple[list[WebSource], list[RagSource]]:
        data = await self.llm_keywords.keywords(question, params)
        queries = []
        school_restrict = params.get("school_domain", False)
        for item in data:
            if not school_restrict:
                queries.append(item["query"])
            else:
                school = item["school"]
                if school.strip() != "":
                    school_domains = self.school_mapper.domains_from_auto(school, 5)
                    print(f"[DOMAINS]", school_domains)
                    if len(school_domains) > 0:
                        queries.append([item["query"], school_domains])
        return await self.pipeline.retrieve(params, queries)

##### vLLM model

In [ ]:
from vllm import SamplingParams, AsyncLLMEngine, AsyncEngineArgs
from vllm.outputs import RequestOutput
from vllm.utils import random_uuid
from vllm.lora.request import LoRARequest
from typing import AsyncGenerator
from typing import Optional, Any
from vllm.transformers_utils.tokenizers import MistralTokenizer
from openai.types.chat import  ChatCompletionUserMessageParam, ChatCompletionSystemMessageParam
from vllm.entrypoints.chat_utils import (
    ChatTemplateContentFormatOption, 
    resolve_chat_template_content_format, 
    apply_hf_chat_template,
    apply_mistral_chat_template,
    parse_chat_messages
)
from vllm.inputs.data import TokensPrompt

class AsyncLLMEngineWrapper:
    """Not support shutdown"""
    def __init__(self) -> None:
        self.engine = None
        self.reap_wait_time = 5
    def init(self, engine_args: AsyncEngineArgs):
        self.engine = AsyncLLMEngine.from_engine_args(engine_args)
    def generate(self, prompt: str | TokensPrompt, sampling_params: SamplingParams, lora_request: LoRARequest | None) -> AsyncGenerator[RequestOutput, None]:
        if self.engine is None:
            raise Exception("Not initialized")
        return self.engine.generate(
            prompt=prompt,
            sampling_params=sampling_params,
            request_id=random_uuid(),
            lora_request=lora_request
        )
    async def chat(self, instruction: str, prompt: str, sampling_params: SamplingParams, lora_request: LoRARequest | None = None):
        messages = [
            ChatCompletionSystemMessageParam(content=instruction, role="system"),
            ChatCompletionUserMessageParam(content=prompt, role="user")
        ]
        return await self._chat(
            messages=messages,
            sampling_params=sampling_params,
            lora_request=lora_request,
            chat_template_kwargs={
                "enable_thinking": False
            }
        )
    async def _chat(
        self,
        messages: list[ChatCompletionUserMessageParam | ChatCompletionUserMessageParam],
        sampling_params: SamplingParams,
        lora_request: LoRARequest | None,
        chat_template_content_format: ChatTemplateContentFormatOption = "auto",
        chat_template: Optional[str] = None,
        add_generation_prompt: bool = True,
        continue_final_message: bool = False,
        chat_template_kwargs: Optional[dict[str, Any]] = None
    ):
        if self.engine is None: raise Exception("Model not loaded")
        tokenizer = await self.engine.get_tokenizer(lora_request)
        model_config = self.engine.engine.get_model_config()
        resolved_content_format = resolve_chat_template_content_format(
            chat_template,
            None,
            chat_template_content_format,
            tokenizer,
            model_config=model_config,
        )
        _chat_template_kwargs: dict[str, Any] = dict(
            chat_template=chat_template,
            add_generation_prompt=add_generation_prompt,
            continue_final_message=continue_final_message,
            tools=None,
        )
        _chat_template_kwargs.update(chat_template_kwargs or {})
        conversation, _ = parse_chat_messages(
            messages, #type:ignore
            model_config,
            tokenizer,
            content_format=resolved_content_format,
        )

        if isinstance(tokenizer, MistralTokenizer):
            prompt_token_ids = apply_mistral_chat_template(
                tokenizer,
                messages=messages, #type:ignore
                **_chat_template_kwargs,
            )
        else:
            prompt_str = apply_hf_chat_template(
                tokenizer=tokenizer, #type:ignore
                conversation=conversation,
                model_config=model_config,
                **_chat_template_kwargs,
            )
            prompt_token_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
        prompt = TokensPrompt(prompt_token_ids=prompt_token_ids)
        return self.generate(
            prompt,
            sampling_params=sampling_params,
            lora_request=lora_request,
        )

##### Client to call model

In [ ]:
import msgspec
class VLLMModel:
    def __init__(self) -> None:
        self._engine = AsyncLLMEngineWrapper()
        self.logger = CmdLogger("Model")
    def init(self, engine_args: AsyncEngineArgs):
        self._engine.init(engine_args)
    async def call(self, call_type: CallType, instruction: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        print(f"[VLLM] {call_type} | Instruction length: {len(instruction)} | Prompt length: {len(prompt)} | kwargs: {params.get('kwargs')}")
        model_id = params["model_id"]
        lora_request = None
        if call_type == CallType.READER and SEP in model_id:
            lora_int_id = int(model_id.split(SEP)[-1])
            lora_request = LORA_MAP.get(lora_int_id)
        sampling_params = msgspec.convert(params, SamplingParams)
        if lora_request != None:
            print(f"[VLLm] Using {lora_request.lora_name}")
        stream = await self._engine.chat(
            instruction=instruction,
            prompt=prompt,
            sampling_params=sampling_params,
            lora_request=lora_request
        )
        total_text = ""
        last_index = 0
        async for event in stream:
            total_text = event.outputs[0].text
            yield total_text[last_index:]
            last_index = len(total_text)
    async def __call__(self, call_type: CallType, instruction: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        return self.call(call_type, instruction, prompt, params)
    async def rerank_page(self, pages: list[SearchResult], query: str, relative_threshold: float, params: GenerationParams) -> list[SearchResult]:
        if len(pages) == 0: return []
        text = ""    
        scores = [0.0 for _ in pages]
        prompt = self._construct_reranker_prompt(query, pages)
        copy_params = copy.deepcopy(params)
        copy_params.update(PAGE_RERANKER_PARAMS) #type:ignore
        async for chunk in await self(
            call_type=CallType.RANKER, 
            instruction=PAGE_RERANKER_INSTRUCTION, 
            prompt=PAGE_RERANKER_PREFIX+prompt, 
            params=copy_params
        ):
            text += chunk
        try:
            if "```" in text:
                text = text.replace("```", "").strip()
            if text[:4] == "json":
                text = text[4:]
            result = json.loads(text)
            if "output" in result:
                for item in result["output"]:
                    index = int(item["index"]) - 1
                    scores[index] = float(item["score"])
            else:
                # Fallback if model not provide intermediate step
                for item in result:
                    index = int(item["index"]) - 1
                    scores[index] = float(item["score"])
        except:
            print(text)
            traceback.print_exc()
        self.logger.log("-----Original-----")
        for page in pages:
            self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
        max_score = 0
        for score, search_result in zip(scores, pages):
            max_score = max(max_score, score)
            search_result["score"] = score
        if max_score == 0: return []
        
        results: list[SearchResult] = []
        for search_result in pages:
            if search_result["score"] >= max_score * relative_threshold:
                results.append(search_result)
        results = sorted(results, key=lambda r:r["score"], reverse=True)
        self.logger.log("-----Reorder-----")
        for page in results:
            self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
        return results
    def _construct_reranker_prompt(self, query: str, data: list[SearchResult]) -> str:
        candidates = [{
            "index": index+1, 
            "title": item["title"],
            "description": item["description"],
        } for index, item in enumerate(data)]
        candidates = "[" + ",\n".join([json.dumps(item, ensure_ascii=False) for item in candidates]) + "]"
        return PAGE_RERANKER_TEMPLATE.format(query=query, pages=candidates)
    async def keywords(self, question: str, params: GenerationParams, threshold: float = 0.5) -> list[KeywordInfo]:
        copy_params = copy.deepcopy(params)
        copy_params.update(KEYWORDS_PARAMS) #type:ignore
        prompt = KEYWORD_TEMPLATE.format(question=question)
        text = ""
        async for chunk in await self(
            call_type=CallType.KEYWORDS, 
            instruction=KEYWORDS_INTRUCTION, 
            prompt=KEYWORDS_PREFIX+prompt, 
            params=copy_params
        ):
            text += chunk
        try:
            if "```" in text:
                text = text.replace("```", "").strip()
            if text[:4] == "json":
                text = text[4:]
            result: list[KeywordInfo] = json.loads(text)
            for item in result:
                self.logger.log(item)
            return result
        except:
            print(text)
            traceback.print_exc()
            return []

##### Pipeline

In [ ]:
class CombinedProtocol(ModelProtocol, KeywordModelProtocol, PageRerankModelProtocol):
    pass
class CustomQA:
    def __init__(self, model_protocol: CombinedProtocol) -> None:
        self.logger = CmdLogger("QA")
        self.retriever = Retriever(model_protocol, model_protocol)
        self.llm_call = model_protocol
    async def start(self):
        await self.retriever.start()
    async def inference(self, prompt: str, request: WorkerChatRequest) -> AsyncGenerator[str, None]:
        text = ""
        async for chunk in await self.llm_call(
            call_type=CallType.READER, 
            instruction=READER_INSTRUCTION, 
            prompt=prompt, 
            params=request["params"]
        ):
            text += chunk
            yield chunk
    async def pre_inference(
        self,
        question: str,
        stream_id: str,
        params: GenerationParams
    ) -> tuple[str, ModelPreOutput]:
        web_sources, rag_sources = await self.retriever.retrive(
            question, 
            params
        )
        context = SourceFormat()(rag_sources)
        prompt = READER_TEMPLATE.format(context=context, question=question)
        self.logger.start()
        pre_output: ModelPreOutput = {
            "generation_params": params,
            "web_sources": web_sources,
            "rag_sources": rag_sources,
            "extra_data": {
            },
            "result_url": stream_id,
        }
        return prompt, pre_output

### Final

VLLMM Engine

In [ ]:
engine_args = AsyncEngineArgs(
    model=MODEL_ID,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.8,
    max_model_len=32768,
    enable_lora=True,
    max_lora_rank=16,
    max_loras=1
)
vllm_model = VLLMModel()
vllm_model.init(engine_args)

Websearch pipeline

In [ ]:
ws_pipeline = CustomQA(vllm_model)
await ws_pipeline.start()
import uuid
REQUEST_STORAGE: dict[str, tuple[str, WorkerChatRequest, ModelPreOutput]] = {}
async def pre_inference_function(request: WorkerChatRequest) -> ModelPreOutput:
    stream_id = str(uuid.uuid4())
    params = request["params"]
    params["school_domain"] = False #True
    params["engine_type"]= "brave"
    params["page_score_threshold"] = 0.51
    params["merge_table"] = True
    prompt, pre_output = await ws_pipeline.pre_inference(
        request["text"],
        stream_id,     
        request["params"]
    ) 
    REQUEST_STORAGE[stream_id] = (prompt, request, pre_output)
    # DataCollector.pre_output(pre_output)
    return pre_output

async def inference_function(stream_id: str) -> AsyncGenerator[str, None]:
    prompt, request, pre_output = REQUEST_STORAGE.pop(stream_id)
    generator = ws_pipeline.inference(prompt, request)
    total = ""
    try:
        async for chunk in generator:
            total += chunk
            yield chunk
    finally:
    # Store chat data when finish
        model_output: ModelOutput = {
            **pre_output,
            "text": total
        }
        data: WorkerStoreChatData = {
            "forward_kwargs": request["forward_kwargs"],
            "model_output": model_output
        }
        # DataCollector.model_output(data)
        # DataCollector.backup(f"{BASE_PATH}data_logs")
        await app.state.store_chat(data)

Connect to server

In [ ]:
app = construct_app(
    server_domain=DOMAIN,
    info=CLIENT_INFO,
    pre_inference=pre_inference_function,
    inferece=inference_function,
    init_tasks=[],
    shutdown_tasks=[],
    is_local=False
)
# CORS policy
from fastapi.middleware.cors import CORSMiddleware
origins = [
    "http://127.0.0.1:8000",
    DOMAIN
]
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)
import nest_asyncio
import uvicorn
nest_asyncio.apply()
uvicorn.run(app, port=NGROK_PORT)

###### Note 
If the instruction is too long -> freeze. Seem like vLLM cache instruction, but we still don't know why it only freeze after some requests. (Seem like cache problem, use llm serve would cause it)
